## 서울시 자치구별 75세 이상 인구 대비 요양시설 정원

본 분석은 `초고령 사회, 서울의 의료·돌봄 인프라는 어디에 더 필요한가? (서울시립대학교 통계학과 이서준, 조성민)`에서 서울시 자치구별 75세 이상 인구 대비 요양시설 정원을 분석하기 위해 작성되었다.

### 1. 분석 기준

- 시설 기준: 국민건강보험공단 장기요양기관 시설별 현황(2024-07-16)(https://www.data.go.kr/data/15124763/fileData.do) 
- 인구 기준: 행정안전부 주민등록 인구통계(2024-07-31)(https://jumin.mois.go.kr/ageStatMonth.do)
- 분석 단위: 서울특별시 25개 자치구
- 주 지표: `입소 정원 ÷ 75세 이상 주민등록인구 × 1,000`
- 보조 지표
  - 잔여정원: `입소 정원 − 현원`
  - 1,000명당 잔여정원: `잔여정원 ÷ 75세 이상 주민등록인구 × 1,000`
  - 가동률: `현원 ÷ 입소 정원`
  - 잔여정원 비율: `잔여정원 ÷ 장기요양 1~2등급 인정자 수 × 100`

`요양시설 입소 정원`의 경우 자료로 `assets/국민건강보험공단_장기요양기관 시설별 현황_20240716.xlsx`를 이용하였으며 다음 기관을 포함한다.

1. 노인요양시설(A01)
2. 노인전문요양시설(A02)
3. 노인요양시설(개정법)(A03)
4. 노인요양공동생활가정(A04)
5. 노인요양시설(단기보호 전환)(A05)

In [14]:
from io import BytesIO
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display, Markdown

pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

plt.rcParams["font.family"] = ["AppleGothic", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

ASSET_DIR = Path("assets")
FACILITY_FILE = next(p for p in ASSET_DIR.glob("*.xlsx") if not p.name.startswith("~$"))

FACILITY_FILE.name

'국민건강보험공단_장기요양기관 시설별 현황_20240716.xlsx'

### 2. 요양시설 정원 집계

`국민건강보험공단_장기요양기관 시설별 현황_20240716.xlsx`에서 `일반현황`의 기관 소재지와 `입소인원`의 유형별 정원을 기관코드로 결합한 뒤 서울 25개 자치구로 집계한다.

소재지를 `기관별 상세주소`에서 추출하여 그 중 시가 `서울특별시`인 경우만을 추출한 뒤 `자치구`, `시설 수`, `요양 시설 정원`, `요양 시설 현원`의 컬럼을 가진 새 표를 만든다.

In [15]:
CARE_FACILITY_TYPES = ["A01", "A02", "A03", "A04", "A05"]

general = pd.read_excel(FACILITY_FILE, sheet_name="일반현황", dtype={"장기요양기관코드": str})
capacity = pd.read_excel(FACILITY_FILE, sheet_name="입소인원", dtype={"장기요양기관코드": str})

# 기관별 상세주소의 첫 두 어절을 시, 자치구로 추출
address = general["기관별 상세주소"].fillna("").str.strip().str.split(n=2, expand=True)
general["시"] = address[0]
general["자치구"] = address[1]

care_capacity = capacity[capacity["기관유형코드"].isin(CARE_FACILITY_TYPES)]

seoul_care = care_capacity.merge(
    general.loc[general["시"] == "서울특별시", ["장기요양기관코드", "자치구"]],
    on="장기요양기관코드",
    how="inner",
    validate="many_to_one",
)

seoul_care_by_gu = (
    seoul_care.groupby("자치구")
    .agg(
        **{
            "시설수": ("장기요양기관코드", "nunique"),
            "요양시설_정원": ("정원", "sum"),
            "요양시설_현원": ("현원", "sum"),
        }
    )
    .astype(int)
    .reset_index()
)

assert len(seoul_care_by_gu) == 25, seoul_care_by_gu["자치구"].tolist()

seoul_care_by_gu.head()

,자치구,시설수,요양시설_정원,요양시설_현원
0,강남구,8,303,253
1,강동구,21,738,690
2,강북구,32,811,679
3,강서구,31,1213,1082
4,관악구,29,627,581


### 3. 75세 이상 주민등록인구 집계

행정안전부 공식 자료를 사용하여 2024년 7월, 75세 이상의 자료를 가져온 뒤 구별로 분류한다.

In [16]:
MOIS_PAGE = "https://jumin.mois.go.kr/ageStatMonth.do"
MOIS_DOWNLOAD = (
    "https://jumin.mois.go.kr/downloadCsvAge.do"
    "?searchYearMonth=month&xlsStats=2"
)

payload = {
    "sltOrgType": "1",
    "sltOrgLvl1": "A",
    "sltOrgLvl2": "",
    "gender": "gender",
    "sum": "sum",
    "sltUndefType": "",  # 전체: 거주자+거주불명자+재외국민, 외국인 제외
    "searchYearStart": "2024",
    "searchMonthStart": "07",
    "searchYearEnd": "2024",
    "searchMonthEnd": "07",
    "sltOrderType": "1",
    "sltOrderValue": "ASC",
    "sltArgTypes": "5",
    "sltArgTypeA": "75",
    "sltArgTypeB": "100",
    "category": "month",
}

session = requests.Session()
session.get(MOIS_PAGE, timeout=30).raise_for_status()
response = session.post(MOIS_DOWNLOAD, data=payload, timeout=120)
response.raise_for_status()
population_raw = pd.read_csv(BytesIO(response.content), encoding="cp949")

population = population_raw.loc[
    population_raw["행정구역"].str.match(r"^서울특별시\s+\S+구\s+\("),
    ["행정구역", "2024년07월_계_연령구간인구수"],
].copy()
population["자치구"] = population["행정구역"].str.extract(
    r"^서울특별시\s+(\S+구)"
)
population["75세이상_인구"] = pd.to_numeric(
    population["2024년07월_계_연령구간인구수"].astype(str).str.replace(",", "", regex=False)
)
population = population[["자치구", "75세이상_인구"]]

assert population["자치구"].nunique() == 25
assert population["75세이상_인구"].sum() == 729_748
population.head()


,자치구,75세이상_인구
1,종로구,13223
2,중구,11441
3,용산구,16985
4,성동구,21146
5,광진구,23391


### 4. 자치구별 공급 수준 계산

In [17]:
analysis = population.merge(seoul_care_by_gu, on="자치구", how="left", validate="one_to_one")
assert len(analysis) == 25 and not analysis.isna().any().any()

analysis["75세이상_천명당_정원"] = (
    analysis["요양시설_정원"] / analysis["75세이상_인구"] * 1_000
)
analysis["정원대비_현원률"] = analysis["요양시설_현원"] / analysis["요양시설_정원"]
analysis["잔여정원"] = analysis["요양시설_정원"] - analysis["요양시설_현원"]
analysis["75세이상_천명당_잔여정원"] = (
    analysis["잔여정원"] / analysis["75세이상_인구"] * 1_000
)
analysis = analysis.sort_values("75세이상_천명당_정원", ascending=False).reset_index(drop=True)
analysis.insert(0, "순위", range(1, len(analysis) + 1))

seoul_rate = analysis["요양시설_정원"].sum() / analysis["75세이상_인구"].sum() * 1_000
seoul_occupancy = analysis["요양시설_현원"].sum() / analysis["요양시설_정원"].sum()
seoul_vacancy_rate = analysis["잔여정원"].sum() / analysis["75세이상_인구"].sum() * 1_000
highest = analysis.iloc[0]
lowest = analysis.iloc[-1]
gap = highest["75세이상_천명당_정원"] / lowest["75세이상_천명당_정원"]

headline = (
    f"실제로 75세 이상 인구 1,000명당 요양시설 정원은 "
    f"{highest['자치구']}가 {highest['75세이상_천명당_정원']:.1f}명인 반면 "
    f"{lowest['자치구']}는 {lowest['75세이상_천명당_정원']:.1f}명으로 "
    f"약 {gap:.1f}배 차이가 난다."
)
display(Markdown(f"### {headline}"))
display(
    analysis[["순위", "자치구", "75세이상_인구", "시설수", "요양시설_정원", "75세이상_천명당_정원", "정원대비_현원률", "잔여정원", "75세이상_천명당_잔여정원"]]
    .style.format({
        "75세이상_인구": "{:,.0f}",
        "시설수": "{:,.0f}",
        "요양시설_정원": "{:,.0f}",
        "75세이상_천명당_정원": "{:.1f}",
        "정원대비_현원률": "{:.1%}",
        "잔여정원": "{:,.0f}",
        "75세이상_천명당_잔여정원": "{:.2f}",
    })
    .background_gradient(subset=["75세이상_천명당_정원"], cmap="Blues")
    .hide(axis="index")
)


### 실제로 75세 이상 인구 1,000명당 요양시설 정원은 금천구가 40.8명인 반면 중구는 7.3명으로 약 5.6배 차이가 난다.

순위,자치구,75세이상_인구,시설수,요양시설_정원,75세이상_천명당_정원,정원대비_현원률,잔여정원,75세이상_천명당_잔여정원
1,금천구,"18,095",22,739,40.8,90.9%,67,3.70
2,도봉구,"30,170",53,"1,162",38.5,92.9%,83,2.75
3,중랑구,"32,216",49,"1,165",36.2,93.9%,71,2.20
4,구로구,"32,154",16,981,30.5,89.1%,107,3.33
5,종로구,"13,223",10,400,30.3,95.5%,18,1.36
6,강서구,"40,264",31,"1,213",30.1,89.2%,131,3.25
7,은평구,"41,074",23,"1,154",28.1,88.1%,137,3.34
8,강북구,"30,874",32,811,26.3,83.7%,132,4.28
9,노원구,"40,834",25,"1,044",25.6,80.2%,207,5.07
10,광진구,"23,391",19,588,25.1,96.6%,20,0.86
